In [2]:
def generate_pairs(column):
    # List to store all the pairs
    pairs = []
    
    for i in range(len(column)):
        for j in range(i, len(column)):  
            if column[i] != '-' and column[j] != '-':  
                pairs.append((column[i], column[j]))
    
    return pairs

In [3]:
test_column = ["A", "T", "G", "G", "G", "A", "A"]
pairs = generate_pairs(test_column)
pairs

[('A', 'A'),
 ('A', 'T'),
 ('A', 'G'),
 ('A', 'G'),
 ('A', 'G'),
 ('A', 'A'),
 ('A', 'A'),
 ('T', 'T'),
 ('T', 'G'),
 ('T', 'G'),
 ('T', 'G'),
 ('T', 'A'),
 ('T', 'A'),
 ('G', 'G'),
 ('G', 'G'),
 ('G', 'G'),
 ('G', 'A'),
 ('G', 'A'),
 ('G', 'G'),
 ('G', 'G'),
 ('G', 'A'),
 ('G', 'A'),
 ('G', 'G'),
 ('G', 'A'),
 ('G', 'A'),
 ('A', 'A'),
 ('A', 'A'),
 ('A', 'A')]

In [5]:
from collections import defaultdict

def count_pairs(alignments):
    pair_counts = defaultdict(int)  
    
    for i in range(len(alignments[0])):  
        column = [seq[i] for seq in alignments]  
        pairs = generate_pairs(column)
        
        for pair in pairs:
            sorted_pair = tuple(sorted(pair))
            pair_counts[sorted_pair] += 1
            
    return pair_counts

In [6]:
alignments = [
    "AGCTACGTGTCGCTGAATCTATGACT", 
    "-GCTA-GAGCA-AGGCAACTGCATCT", 
    "A-CTG-CACCC-ATGAACCTCGCGCT",
    "A-CTG-CACCC-ATGAACCTCTCGCT",
    "A-CTG-CACCC-ATGAACCTCTCGCT",
    "A-CTG-CACCC-ATGAACCTCTCACT",
    "A-CTG-CACCC-ATGAACCTCTCACT"
]
pair_counts = count_pairs(alignments)
pair_counts

defaultdict(int,
            {('A', 'A'): 125,
             ('G', 'G'): 63,
             ('C', 'C'): 205,
             ('T', 'T'): 124,
             ('A', 'G'): 21,
             ('C', 'G'): 31,
             ('A', 'T'): 10,
             ('C', 'T'): 16,
             ('A', 'C'): 33,
             ('G', 'T'): 14})

In [7]:
def calculate_frequencies(alignments):
    total_count = 0
    nucleotide_counts = defaultdict(int)
    
    for seq in alignments:
        for nucleotide in seq:
            if nucleotide != '-': 
                nucleotide_counts[nucleotide] += 1
                total_count += 1
    
    frequencies = {nuc: count / total_count for nuc, count in nucleotide_counts.items()}
    
    return frequencies

In [8]:
frequencies = calculate_frequencies(alignments)
frequencies

{'A': 0.24390243902439024,
 'G': 0.15853658536585366,
 'C': 0.3780487804878049,
 'T': 0.21951219512195122}

In [9]:
import math

def calculate_scores(pair_counts, freqs, scale=3):
    total_pairs = sum(pair_counts.values())
    scores = {}
    
    for (x, y), count in pair_counts.items():
        observed_freq = count / total_pairs
        expected_freq = freqs[x] * freqs[y]
        
        score = scale * math.log2(observed_freq / expected_freq)
        scores[(x, y)] = round(score)
    
    return scores

In [10]:
scores = calculate_scores(pair_counts, frequencies)
scores

{('A', 'A'): 5,
 ('G', 'G'): 6,
 ('C', 'C'): 3,
 ('T', 'T'): 6,
 ('A', 'G'): -1,
 ('C', 'G'): -1,
 ('A', 'T'): -5,
 ('C', 'T'): -5,
 ('A', 'C'): -3,
 ('G', 'T'): -2}

In [16]:
def create_blosum_matrix(scores, nucleotides):
    blosum_matrix = {nuc: {} for nuc in nucleotides}
    
    for (x, y), score in scores.items():
        blosum_matrix[x][y] = score
        blosum_matrix[y][x] = score  

    return blosum_matrix

In [17]:
scores = {('A', 'A'): 5, ('G', 'G'): 5, ('C', 'C'): 3, ('T', 'T'): 6, 
          ('A', 'G'): 1, ('C', 'G'): 0, ('A', 'T'): -4, ('C', 'T'): -4, 
          ('A', 'C'): -1, ('G', 'T'): -1}
nucleotides = ['A', 'G', 'C', 'T']

blosum_matrix = create_blosum_matrix(scores, nucleotides)
blosum_matrix

{'A': {'A': 5, 'G': 1, 'T': -4, 'C': -1},
 'G': {'G': 5, 'A': 1, 'C': 0, 'T': -1},
 'C': {'C': 3, 'G': 0, 'T': -4, 'A': -1},
 'T': {'T': 6, 'A': -4, 'C': -4, 'G': -1}}

In [18]:
def print_blosum_matrix(matrix, nucleotides):
    print("   " + "  ".join(nucleotides))
    
    for nuc in nucleotides:
        row = [str(matrix[nuc][x]).rjust(3) for x in nucleotides]
        print(f"{nuc} {' '.join(row)}")

print_blosum_matrix(blosum_matrix, nucleotides)

   A  G  C  T
A   5   1  -1  -4
G   1   5   0  -1
C  -1   0   3  -4
T  -4  -1  -4   6


In [19]:
def init(rows, cols, gap_penalty=10):
    dp_matrix = [[0 for _ in range(cols + 1)] for _ in range(rows + 1)]
    
    for i in range(1, rows + 1):
        dp_matrix[i][0] = -i * gap_penalty
    for j in range(1, cols + 1):
        dp_matrix[0][j] = -j * gap_penalty

    return dp_matrix

print(init(3, 3, 4))

[[0, -4, -8, -12], [-4, 0, 0, 0], [-8, 0, 0, 0], [-12, 0, 0, 0]]


In [20]:
def get_new_score(up, left, middle, s_score, gap_penalty):
    match = middle + s_score
    insert = left - gap_penalty
    delete = up - gap_penalty
    
    return max(match, insert, delete)

print(get_new_score(0, 10, 2, 0, 2))
print(get_new_score(-16, -7, -14, 0, 2))

8
-9


In [21]:
def align(top_seq, bottom_seq, gap_penalty, blosum_matrix):
    rows = len(top_seq)
    cols = len(bottom_seq)
    
    dp_matrix = init(rows, cols, gap_penalty)
    
    for i in range(1, rows + 1):
        for j in range(1, cols + 1):
            s_score = blosum_matrix[top_seq[i-1]][bottom_seq[j-1]]
            dp_matrix[i][j] = get_new_score(dp_matrix[i-1][j], dp_matrix[i][j-1], dp_matrix[i-1][j-1], s_score, gap_penalty)
    
    return dp_matrix

top_seq = "AGTACGCA"
bottom_seq = "TATGC"
gap_penalty = 2
alignment_matrix = align(top_seq, bottom_seq, gap_penalty, blosum_matrix)
alignment_matrix

[[0, -2, -4, -6, -8, -10],
 [-2, -4, 3, 1, -1, -3],
 [-4, -3, 1, 2, 6, 4],
 [-6, 2, 0, 7, 5, 3],
 [-8, 0, 7, 5, 8, 6],
 [-10, -2, 5, 3, 6, 11],
 [-12, -4, 3, 4, 8, 9],
 [-14, -6, 1, 2, 6, 11],
 [-16, -8, -1, 0, 4, 9]]

In [22]:
def get_alignment(top_seq, bottom_seq, sm, gap_penalty, blosum_matrix):
    i, j = len(top_seq), len(bottom_seq)
    aligned_top = []
    aligned_bottom = []
    
    while i > 0 and j > 0:
        s_score = blosum_matrix[top_seq[i-1]][bottom_seq[j-1]]
        if sm[i][j] == sm[i-1][j-1] + s_score:  
            aligned_top.append(top_seq[i-1])
            aligned_bottom.append(bottom_seq[j-1])
            i -= 1
            j -= 1
        elif sm[i][j] == sm[i][j-1] - gap_penalty: 
            aligned_top.append('-')
            aligned_bottom.append(bottom_seq[j-1])
            j -= 1
        else:  
            aligned_top.append(top_seq[i-1])
            aligned_bottom.append('-')
            i -= 1
    
    while i > 0:
        aligned_top.append(top_seq[i-1])
        aligned_bottom.append('-')
        i -= 1
    while j > 0:
        aligned_top.append('-')
        aligned_bottom.append(bottom_seq[j-1])
        j -= 1
    
    
    return ''.join(reversed(aligned_top)), ''.join(reversed(aligned_bottom))

aligned_top, aligned_bottom = get_alignment(top_seq, bottom_seq, alignment_matrix, gap_penalty, blosum_matrix)
print(aligned_top)
print(aligned_bottom)

AGTACGCA
--TATGC-
